# Open Bandit Dataset RL-style EDA

## Problem Statement
This notebook performs complete EDA on logged bandit interaction data before reinforcement-learning-style modeling.

## What this notebook covers
- OBP dataset loading
- flattening logged bandit feedback
- missing values
- duplicates
- reward analysis
- action analysis
- policy analysis
- propensity score analysis
- context feature analysis
- cleaned CSV save
- modeling table save
- PCA / t-SNE context visualization

## Telugu
Bandit / RL modeling mundu logged interaction data ni deep ga ardham chesukovali.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

from obp.dataset import OpenBanditDataset

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

sns.set_theme(style="whitegrid")

os.makedirs("../data/raw", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../reports/figures", exist_ok=True)
os.makedirs("../reports", exist_ok=True)

## Folder structure expected

```text
eda-open-bandit-rl/
├── data/
│   ├── raw/
│   │   └── obd/
│   └── processed/
├── notebooks/
│   └── open_bandit_eda.ipynb
├── reports/
│   ├── figures/
│   └── insights.md
└── README.md
```

In [ ]:
behavior_policy = "random"   # or "bts"
campaign = "all"             # or "men" / "women"

dataset = OpenBanditDataset(
    behavior_policy=behavior_policy,
    campaign=campaign,
    data_path="../data/raw/obd"
)

bandit_feedback = dataset.obtain_batch_bandit_feedback()

print("Bandit feedback keys:", bandit_feedback.keys())
print("Number of rounds:", bandit_feedback["n_rounds"])
print("Number of actions:", bandit_feedback["n_actions"])

## Why this loader?
Official OBP API use chesthe logged bandit feedback clean ga dorukuthundi.

In [ ]:
n_rounds = bandit_feedback["n_rounds"]

df_bandit = pd.DataFrame({
    "action": bandit_feedback["action"],
    "reward": bandit_feedback["reward"],
    "pscore": bandit_feedback["pscore"],
})

if "position" in bandit_feedback:
    df_bandit["position"] = bandit_feedback["position"]

if "context" in bandit_feedback:
    context = bandit_feedback["context"]
    context_cols = [f"context_{i}" for i in range(context.shape[1])]
    df_context = pd.DataFrame(context, columns=context_cols)
    df_bandit = pd.concat([df_bandit, df_context], axis=1)
else:
    context_cols = []

df_bandit["behavior_policy"] = behavior_policy
df_bandit["campaign"] = campaign

print("Flat bandit table shape:", df_bandit.shape)
display(df_bandit.head())

In [ ]:
df_bandit.to_csv("../data/processed/bandit_raw_export.csv", index=False)
print("Saved: ../data/processed/bandit_raw_export.csv")

In [ ]:
print(df_bandit.info())
print("\n")
display(df_bandit.describe(include="all").T)

In [ ]:
missing_report = (
    df_bandit.isna()
      .sum()
      .to_frame("missing_count")
      .assign(missing_pct=lambda x: (x["missing_count"] / len(df_bandit) * 100).round(2))
      .sort_values("missing_pct", ascending=False)
)

display(missing_report)
missing_report.to_csv("../data/processed/bandit_missing_report.csv")

In [ ]:
duplicate_count = df_bandit.duplicated().sum()
print("Duplicate rows:", duplicate_count)

In [ ]:
df_bandit = df_bandit.drop_duplicates().copy()
print("Shape after duplicate removal:", df_bandit.shape)

In [ ]:
num_cols = df_bandit.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df_bandit.select_dtypes(include=["object"]).columns.tolist()

print("Numerical columns:", num_cols)
print("Categorical columns:", cat_cols)

# Reward and Action Analysis
Bandit data lo rewards usually sparse untayi.  
Actions frequency and pscore behavior chudali.

In [ ]:
reward_summary = pd.DataFrame([{
    "reward_mean": float(df_bandit["reward"].mean()),
    "reward_sum": float(df_bandit["reward"].sum()),
    "reward_positive_rate": float((df_bandit["reward"] > 0).mean()),
    "n_rows": int(len(df_bandit))
}])

display(reward_summary)
reward_summary.to_csv("../data/processed/bandit_reward_summary.csv", index=False)

In [ ]:
action_summary = (
    df_bandit.groupby("action", as_index=False)
             .agg(
                 action_count=("action", "size"),
                 avg_reward=("reward", "mean"),
                 total_reward=("reward", "sum"),
                 avg_pscore=("pscore", "mean")
             )
             .sort_values("action_count", ascending=False)
)

display(action_summary.head(20))
action_summary.to_csv("../data/processed/bandit_action_summary.csv", index=False)

In [ ]:
policy_summary = (
    df_bandit.groupby(["behavior_policy", "campaign"], as_index=False)
             .agg(
                 n_rows=("reward", "size"),
                 avg_reward=("reward", "mean"),
                 avg_pscore=("pscore", "mean")
             )
)

display(policy_summary)
policy_summary.to_csv("../data/processed/bandit_policy_summary.csv", index=False)

In [ ]:
numeric_summary = []

for col in num_cols:
    arr = df_bandit[col].dropna().to_numpy()
    if arr.size == 0:
        continue

    numeric_summary.append({
        "column": col,
        "count": arr.size,
        "mean": float(np.mean(arr)),
        "median": float(np.median(arr)),
        "std": float(np.std(arr)),
        "var": float(np.var(arr)),
        "min": float(np.min(arr)),
        "max": float(np.max(arr)),
        "p01": float(np.percentile(arr, 1)),
        "p05": float(np.percentile(arr, 5)),
        "p25": float(np.percentile(arr, 25)),
        "p50": float(np.percentile(arr, 50)),
        "p75": float(np.percentile(arr, 75)),
        "p95": float(np.percentile(arr, 95)),
        "p99": float(np.percentile(arr, 99))
    })

bandit_numeric_summary = pd.DataFrame(numeric_summary).sort_values("column")
display(bandit_numeric_summary)
bandit_numeric_summary.to_csv("../data/processed/bandit_numeric_summary.csv", index=False)

In [ ]:
bandit_cat_summary = []

for col in cat_cols:
    mode_series = df_bandit[col].mode(dropna=True)
    bandit_cat_summary.append({
        "column": col,
        "n_unique": int(df_bandit[col].nunique(dropna=True)),
        "most_frequent": mode_series.iloc[0] if len(mode_series) > 0 else np.nan,
        "missing_count": int(df_bandit[col].isna().sum()),
        "missing_pct": round(df_bandit[col].isna().mean() * 100, 2)
    })

bandit_categorical_summary = pd.DataFrame(bandit_cat_summary).sort_values("n_unique", ascending=False)
display(bandit_categorical_summary)
bandit_categorical_summary.to_csv("../data/processed/bandit_categorical_summary.csv", index=False)

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(data=df_bandit, x="reward")
plt.title("Reward Distribution")
plt.tight_layout()
plt.savefig("../reports/figures/reward_distribution.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
top_actions = action_summary.head(20)

plt.figure(figsize=(12, 5))
sns.barplot(data=top_actions, x="action", y="action_count")
plt.title("Top Actions by Frequency")
plt.tight_layout()
plt.savefig("../reports/figures/action_frequency.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
sns.barplot(data=policy_summary, x="behavior_policy", y="avg_reward")
plt.title("Average Reward by Behavior Policy")
plt.tight_layout()
plt.savefig("../reports/figures/reward_by_policy.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df_bandit["pscore"], bins=40, kde=True)
plt.title("Propensity Score Distribution")
plt.tight_layout()
plt.savefig("../reports/figures/pscore_distribution.png", dpi=200, bbox_inches="tight")
plt.show()

## Why pscore analysis?
`pscore` = logged policy probability for selected action.  
Later IPS / SNIPS / DR laanti off-policy evaluation methods ki idi key input.

In [ ]:
if len(context_cols) > 1:
    plt.figure(figsize=(12, 8))
    sns.heatmap(df_bandit[context_cols].corr(numeric_only=True), cmap="coolwarm", center=0)
    plt.title("Context Correlation Heatmap")
    plt.tight_layout()
    plt.savefig("../reports/figures/context_correlation_heatmap.png", dpi=200, bbox_inches="tight")
    plt.show()

In [ ]:
reward_by_action = (
    df_bandit.groupby("action", as_index=False)
             .agg(
                 avg_reward=("reward", "mean"),
                 avg_pscore=("pscore", "mean"),
                 count=("reward", "size")
             )
             .sort_values("avg_reward", ascending=False)
)

display(reward_by_action.head(20))

In [ ]:
df_clean = df_bandit.copy()

In [ ]:
for col in cat_cols:
    if df_clean[col].isna().sum() > 0:
        df_clean[col] = df_clean[col].fillna("Unknown")

for col in num_cols:
    if df_clean[col].isna().sum() > 0:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

In [ ]:
final_missing = df_clean.isna().sum().sort_values(ascending=False)
display(final_missing.head(20))
print("Total remaining missing values:", int(final_missing.sum()))

In [ ]:
df_clean.to_csv("../data/processed/bandit_cleaned.csv", index=False)
print("Saved: ../data/processed/bandit_cleaned.csv")

In [ ]:
model_cols = ["action", "reward", "pscore"] + context_cols

if "position" in df_clean.columns:
    model_cols.append("position")

model_table = df_clean[model_cols].copy()

display(model_table.head())
print("Model table shape:", model_table.shape)

model_table.to_csv("../data/processed/bandit_model_table.csv", index=False)
print("Saved: ../data/processed/bandit_model_table.csv")

## Why separate model table?
Bandit / RL-style modeling ki mainly kavalsindi:
- context
- action
- reward
- pscore

In [ ]:
if len(context_cols) > 0:
    scaler = StandardScaler()
    X_context = scaler.fit_transform(df_clean[context_cols])
    print("Scaled context shape:", X_context.shape)

In [ ]:
if len(context_cols) > 1:
    pca = PCA(n_components=2, random_state=42)
    X_pca = pca.fit_transform(X_context)

    pca_df = pd.DataFrame(X_pca, columns=["pc1", "pc2"])
    pca_df["action"] = df_clean["action"].values
    pca_df["reward"] = df_clean["reward"].values

    print("Explained variance ratio:", pca.explained_variance_ratio_)
    display(pca_df.head())

In [ ]:
if len(context_cols) > 1:
    fig = px.scatter(
        pca_df.sample(min(len(pca_df), 5000), random_state=42),
        x="pc1",
        y="pc2",
        color="reward",
        hover_data=["action"],
        title="PCA View of Context Space"
    )
    fig.write_html("../reports/figures/pca_context_view.html")
    fig.show()

In [ ]:
if len(context_cols) > 1:
    sample_n = min(len(df_clean), 3000)
    sampled = df_clean.sample(sample_n, random_state=42)
    X_sample = scaler.transform(sampled[context_cols])

    tsne = TSNE(n_components=2, random_state=42, perplexity=30, init="pca", learning_rate="auto")
    X_tsne = tsne.fit_transform(X_sample)

    tsne_df = pd.DataFrame(X_tsne, columns=["tsne1", "tsne2"])
    tsne_df["action"] = sampled["action"].values
    tsne_df["reward"] = sampled["reward"].values

    fig = px.scatter(
        tsne_df,
        x="tsne1",
        y="tsne2",
        color="reward",
        hover_data=["action"],
        title="t-SNE View of Context Space"
    )
    fig.write_html("../reports/figures/tsne_context_view.html")
    fig.show()

In [ ]:
insights = f'''
# Open Bandit Dataset EDA Insights

1. This project uses logged bandit feedback from the Open Bandit Dataset under behavior policy `{behavior_policy}` and campaign `{campaign}`.
2. The flattened table contains action, reward, propensity score, and context features suitable for offline bandit analysis.
3. Reward distribution should be checked carefully because bandit rewards are often sparse.
4. Action frequency is not uniform, so logged behavior must be interpreted together with propensity scores.
5. Propensity scores are essential for later off-policy evaluation methods such as IPS/SNIPS/DR.
6. Context features were extracted and organized into a modeling-ready table.
7. A cleaned logged dataset and a final model table were saved before reinforcement-learning-style modeling.
8. PCA and t-SNE views provide a useful first look at the structure of the context space.
'''

with open("../reports/insights.md", "w", encoding="utf-8") as f:
    f.write(insights)

print(insights)